In [1]:
master = "local"
dataFolder = "/compar-spark-docker/databases/" if master == "spark://spark-master:7077" else "../databases/"

In [2]:
import sys
sys.path.append("../src")

from pyspark.sql import SparkSession
spark = (SparkSession.builder
    .master("local[*]")
    .appName("dev-m1")
    .config("spark.driver.memory", "2g")
    .getOrCreate())

df = spark.read.parquet("../databases/yellow_tripdata_2025-01.parquet")
df.printSchema()
print(df.count())

zones = spark.read.option("header", True).csv("../databases/taxi_zone_lookup.csv")
zones.show(5)

c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

3475226
+----------+-------------+--------------------+------------+
|Locat

In [3]:
import sys; sys.path.append("../src")
import main
trips = main.load_trips(spark, "../databases", [2025], 1.0)
trips.printSchema()
print(trips.count())

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)

19348004


In [4]:
from pyspark.sql import functions as F
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|         540149|            0|    540149|            540149|           0|    

In [5]:

from pyspark.sql import functions as F
import sys, importlib
sys.path.insert(0, "../src")
import main, m1
importlib.reload(main); importlib.reload(m1)

flagged= m1.flag_invalid(trips)
stats= m1.cleaning_stats(flagged)
stats.show()

tripsClean= m1.clean(flagged)
cleanCount= tripsClean.count()
print("Rows cleaned: ", cleanCount)


row= stats.agg(F.sum("total").alias("t"), F.sum("removed_any").alias("r")).first()
print("total-removed: ", row.t - row.r, "\n Cleaned: ", cleanCount)

leftovers= tripsClean.filter(
    (F.col("fare_amount")<= 0) | (F.col("trip_distance")<=0) |
    (F.col("passenger_count")==0 ) | (F.col("passenger_count")>6)|
    (F.col("tpep_dropoff_datetime")<= F.col("tpep_pickup_datetime"))|
    (~F.col("PULocationID").between(1, 263)) |
    (~F.col("DOLocationID").between(1, 263))
).count()
print("invalid rown left: ", leftovers)

print("null passanger_count kept: ", tripsClean.filter(F.col("passenger_count").isNull()).count())

+----+--------+------------+------------+-----------+------------+------------+--------------+------------+-----------+
|year|   total|removed_fare|removed_dist|removed_pax|removed_time|removed_zone|removed_period|removed_null|removed_any|
+----+--------+------------+------------+-----------+------------+------------+--------------+------------+-----------+
|2007|       1|           0|           0|          0|           0|           0|             0|           0|          0|
|2009|       2|           0|           0|          0|           0|           0|             0|           0|          0|
|2024|      21|           0|           0|          0|           0|           0|             0|           0|          0|
|2025|19347980|     1042872|      542643|     111923|      144203|      145768|             0|           0|    1864327|
+----+--------+------------+------------+-----------+------------+------------+--------------+------------+-----------+

Rows cleaned:  17483677
total-removed: 